In [103]:
import pandas as pd
import numpy as np
import re
pd.set_option('display.max_columns', None)

In [104]:
df = pd.read_csv("../../datasets/raw/steam_games_requirements.csv")

In [105]:
df.head(1)

,Unnamed: 0,url,types,name,desc_snippet,recent_reviews,all_reviews,release_date,developer,publisher,popular_tags,game_details,languages,achievements,genre,game_description,mature_content,minimum_requirements,recommended_requirements,original_price,discount_price
0,0,https://store.steampowered.com/app/379720/DOOM/,app,DOOM,Now includes all three premium DLC packs (Unto...,"Very Positive,(554),- 89% of the 554 user revi...","Very Positive,(42,550),- 92% of the 42,550 use...","May 12, 2016",id Software,"Bethesda Softworks,Bethesda Softworks","FPS,Gore,Action,Demons,Shooter,First-Person,Gr...","Single-player,Multi-player,Co-op,Steam Achieve...","English,French,Italian,German,Spanish - Spain,...",54.0,Action,"About This Game Developed by id software, the...",NaN,"Minimum:,OS:,Windows 7/8.1/10 (64-bit versions...","Recommended:,OS:,Windows 7/8.1/10 (64-bit vers...",$19.99,$14.99


In [106]:
df = df[df['types'] == 'app']
df.drop(columns=['types'], inplace=True)

In [107]:
df['url'] = df['url'].apply(lambda row: row.split('/')[4])

In [108]:
df.rename(columns={'url': 'app_id'}, inplace=True)

In [109]:
df = df[['app_id', 'name', 'minimum_requirements', 'recommended_requirements']]

In [110]:
df.head(1)

,app_id,name,minimum_requirements,recommended_requirements
0,379720,DOOM,"Minimum:,OS:,Windows 7/8.1/10 (64-bit versions...","Recommended:,OS:,Windows 7/8.1/10 (64-bit vers..."


In [111]:
df.isna().sum()

app_id                          0
name                           14
minimum_requirements        16952
recommended_requirements    16946
dtype: int64

In [116]:
df[df[['minimum_requirements', 'recommended_requirements']].isna().all(axis=1)]

,app_id,name,minimum_requirements,recommended_requirements
16,597170,Clone Drone in the Danger Zone,NaN,NaN
20,42700,Call of Duty®: Black Ops,NaN,NaN
24,12210,Grand Theft Auto IV,NaN,NaN
26,400,Portal,NaN,NaN
28,704450,Neverwinter Nights: Enhanced Edition,NaN,NaN
...,...,...,...,...
40813,912210,Achievement Collector: Cat,NaN,NaN
40815,912140,SpaceBall in Cube,NaN,NaN
40824,906470,Gravia,NaN,NaN
40826,906430,Alive,NaN,NaN


In [113]:
df.to_csv("../../datasets/modifiedHF.csv")

In [81]:
def extract_cpu(strings):
    if pd.isna(strings):
        return None
    if 'Processor:' in strings:
        s = strings.split(',')
        return s[s.index('Processor:')+1]
    return None

In [82]:
req = df[df['minimum_requirements'].str.contains('Processor|CPU', case=True)]['minimum_requirements'].tolist()

In [83]:
def extract_cpu2(text):
    """Extract CPU info from Processor line."""
    CPU_LINE_PATTERN = r'(?i)(?:Processor|CPU):[,\s]+([^,]+)'
    if pd.isna(text):
        return None
    text = str(text)
    match = re.search(CPU_LINE_PATTERN, text)
    if match:
        return match.group(1).strip()
    return None

In [84]:
extract_cpu2(req[110])

'Core i5-760 or better / AMD Phenom II X4 or better [Quad-core CPU]'

In [85]:
df['rec_cpu'] = df['recommended_requirements'].apply(lambda x: extract_cpu2(x))

In [86]:
df['min_cpu'] = df['minimum_requirements'].apply(lambda x: extract_cpu2(x))

In [87]:
df[df['rec_cpu'].isna()]

,app_id,name,minimum_requirements,recommended_requirements,rec_cpu,min_cpu
13,393080,Call of Duty®: Modern Warfare® Remastered,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",NaN,Intel Core i3-3225 @ 3.30GHz or equivalent
33,638970,Yakuza 0,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",NaN,Intel Core i5-3470 | AMD FX-6300
58,799640,Dungeon Munchies,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",NaN,Intel Core i5
62,364470,The Elder Scrolls®: Legends™,"Minimum:,OS:,Windows 7 / Windows 8 / Windows 1...","Recommended:,Additional Notes:,Keyboard and mouse",NaN,Intel Pentium D or AMD® Athlon™ 64 X2
65,813820,Realm Royale,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",NaN,Intel(R) Core(TM) i5-2320 CPU @ 3.00 GHz (4 CPUs)
...,...,...,...,...,...,...
40741,894620,ATONE: Heart of the Elder Tree,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",NaN,Intel Core i3 3217U 1.8 GHZ Dual Core
40778,703860,GRID,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",NaN,TBC
40792,915430,Triteckka: The pure shooter,"Minimum:,OS:,Windows 10,Processor:,Dual Core C...","Recommended:,DirectX:,Version 10",NaN,Dual Core CPU
40804,909470,Touch Type Tale - Strategic Typing,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",NaN,NaN


In [88]:
df['min_cpu'].isna().sum()

np.int64(1139)